# Milvus + Woodpecker: full-stack test walkthrough

Run this notebook from top to bottom with **Shift+Enter**, or use **Run All** after reviewing the configuration.
It invokes the existing `stack.sh` commands and uses the same dashboards, alerts and performance runner as the CLI.

| Stage | What you will see |
| --- | --- |
| Prepare and install | Target namespace/context, rendered charts and ready Pods |
| Open endpoints | Four E2E dashboards, three reference dashboards, Prometheus and Alertmanager |
| Write smoke | A new collection, inserts, primary-key query and vector search |
| Verify monitoring | Scrape health, all dashboard queries and alert coverage gaps |
| Verify alert pipeline | Offline rule tests, firing, AM delivery, recovery and inhibition |
| Run write performance | Throughput, p50/p95/p99, correctness, latency plot, metrics and logs |
| Diagnose and collect | Slow requests, timestamped Pod logs and machine-readable evidence |
| Pause or reset | Separate opt-in cells for stop/start and uninstall/install |

Configure an existing Kubernetes cluster and install Helm 3, kubectl and curl first; the notebook runtime uses Python 3.11+. Resource/workload changes default to previews until EXECUTE=True.
Allow several minutes for deployment, scrape windows, rule reloads and performance collection.
Email is disabled. Alerts are visible in Prometheus and Alertmanager.

**Run All leaves the stack running.** Stop/uninstall cells are disabled by default. `uninstall` deletes this stack's persistent data.

In [ ]:
from pathlib import Path
import html
import json
import os
import shutil
import subprocess
import sys
import time
from IPython.display import HTML, Markdown, display

candidates = []
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates.extend([parent, parent / "deployments" / "milvus-full-stack-helm"])
ROOT = next((p for p in candidates if (p / "stack.sh").is_file() and (p / "scripts" / "perf.py").is_file()), None)
assert ROOT is not None, "Open this notebook from inside the repository"
print("Stack directory:", ROOT)

## 1. Select an existing Kubernetes context and review execution

Configure your own Kubernetes first, whether it is a shared cluster, kind or minikube. `stack.sh` never creates, starts or stops Kubernetes. Use `kubectl config current-context` and check connectivity/credentials before proceeding.

`CONTEXT` defaults to kubectl's current context. `NAMESPACE` scopes this stack. Review the image and load settings below; private ARM64 images require registry access or externally prepared node caches.

**`EXECUTE=False` is the default:** resource changes, forwards and workloads are printed for review. Set **`EXECUTE=True`** explicitly to run them (`-x` in the CLI). Read-only discovery and local render preparation may run in preview mode. All six ports must belong to this stack; existing matching forwards are reused.


In [ ]:
CONTEXT = os.getenv("CONTEXT", "")  # Empty uses kubectl current-context.
NAMESPACE = os.getenv("NAMESPACE", "milvus-stack")
PORTS = {key: int(os.getenv(key + "_PORT", default)) for key, default in {
    "GRAFANA": 3000, "PROMETHEUS": 9099, "MILVUS": 19530, "ALERTMANAGER": 9093, "JAEGER": 16686, "LOKI": 3100}.items()}
USE_LOCAL_ARM64 = True
EXECUTE = False  # Review previews before opting in to resource/workload changes.
RUN_INSTALL = True
RUN_RULE_TESTS = True
RUN_ALERT_PROBE = True
RUN_PERF = True
RUN_GATE_DEMO = False
RUN_ACCEPTANCE = False  # Repeats the earlier tests; useful as a single CI command.

SMOKE_SETTINGS = {"BATCHES": "3", "BATCH_SIZE": "100", "INTERVAL": "2"}
PERF_SETTINGS = {"PERF_DURATION": "60", "PERF_WARMUP": "5", "PERF_COOLDOWN": "35",
                 "PERF_CONCURRENCY": "2", "PERF_BATCH_SIZE": "100", "PERF_DIMENSION": "128",
                 "PERF_RPS": "5", "PERF_MIN_ROWS_PER_SECOND": "0", "PERF_MAX_P99_MS": "0"}


In [ ]:
if "lab" in globals():
    lab.close_forwards()  # Only closes forwards that this notebook created.
sys.path.insert(0, str(ROOT / "scripts"))
import importlib
import notebook_helpers
importlib.reload(notebook_helpers)
from notebook_helpers import StackLab, show_report
lab = StackLab(ROOT, CONTEXT, NAMESPACE, PORTS, USE_LOCAL_ARM64, execute=EXECUTE)
print(json.dumps({"context": lab.context, "execute": EXECUTE, "namespace": NAMESPACE,
                  "ports": PORTS, "local_arm64": USE_LOCAL_ARM64}, indent=2))
required = ["helm", "kubectl", "curl"]
missing = [name for name in required if not shutil.which(name)]
assert not missing, f"Install these tools first: {missing}"
print("Required tools found. Notebook commands stream output and fail on nonzero exit status.")
lab.run("help")

## 2. Inspect the existing environment and install

This full walkthrough selects `--with all` for render/images/install. The CLI default selects only Milvus + WP + etcd/MinIO. Addons can be installed later independently.

`install` wraps the official Helm install/upgrade commands. `start/stop` only resume/pause this stack's existing Kubernetes workloads; `start` does not install a missing stack. `uninstall` uninstalls releases and deletes their PVC data.

Resource/workload commands print plans unless `EXECUTE=True`. Local render output is under `.local/rendered/`; `images` lists the required images without loading them into nodes. Prepare image access outside this tool. Commands and plans are retained under `.local/` for review.


In [ ]:
lab.run("status")  # Read-only: validate the existing context and inspect releases.


In [ ]:
lab.run("render")
lab.run("images")  # Inventory only. Prepare registry access/node image caches separately.


In [ ]:
if RUN_INSTALL:
    lab.run("install")
lab.run("status")

## 3. Open dashboards and services

In preview mode the next cell prints native port-forward commands. With `EXECUTE=True`, it starts only missing forwards and reuses matching existing listeners. It never manages the Kubernetes cluster. Unrelated occupied ports are refused.

Grafana login: **admin / admin**. Keep the kernel/forwards running while viewing dashboards. Forward logs live in `.local/notebook/forward.log`.


In [ ]:
lab.open_forwards()
grafana = lab.env["GRAFANA_URL"]
links = {"E2E Write / Append": grafana + "/d/milvus-e2e-write",
         "E2E WAL Read / Consume": grafana + "/d/milvus-e2e-consume",
         "E2E Flush / Checkpoint": grafana + "/d/milvus-e2e-flush",
         "E2E Search / Query": grafana + "/d/milvus-e2e-query",
         "Milvus dashboard": grafana + "/d/milvus-full-stack",
         "Woodpecker client dashboard": grafana + "/d/woodpecker-client-k8s",
         "Woodpecker server dashboard": grafana + "/d/woodpecker-server-k8s",
         "Prometheus alerts": lab.env["PROMETHEUS_URL"] + "/alerts",
         "Alertmanager": lab.env["ALERTMANAGER_URL"], "Jaeger": lab.env["JAEGER_URL"],
         "Loki API": lab.env["LOKI_URL"], "Milvus REST": lab.env["MILVUS_URL"]}
display(HTML("<table><tr><th>Service</th><th>Local URL</th></tr>" + "".join(
    '<tr><td>' + html.escape(name) + '</td><td><a target="_blank" href="' + html.escape(url, quote=True) + '">' + html.escape(url) + '</a></td></tr>'
    for name, url in links.items()) + "</table>"))

## 4. Insert a small workload and verify it

The default smoke inserts **300 eight-dimensional vectors** in three batches.
Every batch checks an ID query and nearest-vector search; the workload then explicitly flushes and checks the complete row count and search again. A uniquely named collection is retained for inspection.
The final event should say `stage: passed`. Results are in `.local/workload.jsonl`.

In [ ]:
if EXECUTE:
    lab.run("smoke", env=SMOKE_SETTINGS)
    smoke_events = [json.loads(line) for line in (lab.state / "workload.jsonl").read_text().splitlines()]
    assert smoke_events[-1]["stage"] == "passed"
    display(smoke_events[-1])
else:
    lab.run('smoke')


## 5. Check collection, dashboards and alert coverage

Wait for two 15-second scrapes after writing, then check all seven dashboards, their PromQL expressions and required E2E stage metrics.
Queries with no samples are recorded separately from query errors: this workload does not exercise every feature or error counter.
The alert report distinguishes rule evaluation health from missing metric/history coverage. C2 needs seven-day history; C6 read-frontier instrumentation may be absent in the chosen image.
The readiness barrier waits for eight business targets and all reference rules to be healthy on two consecutive checks. It tolerates brief scrape/rollout transitions, but raises an error after 180 seconds if they persist.


In [ ]:
if EXECUTE:
    print("Waiting 35 seconds for the final writes to reach Prometheus...")
    time.sleep(35)
    lab.wait_monitoring(timeout=180)
    lab.run("verify")
    verification = json.loads((lab.state / "verification.json").read_text())
    display({uid: {key: value for key, value in report.items() if key in ("queries", "queries_with_data", "errors")}
             for uid, report in verification["dashboards"].items()})
else:
    print("Preview mode: this verification/evidence step follows the explicitly executed workload.")


In [ ]:
if EXECUTE:
    lab.run("alerts")
    alert_report = json.loads((lab.state / "alerting.json").read_text())
    gaps = [{key: row.get(key) for key in ("rule_id", "missing_metric_names", "seven_day_history_available")}
            for row in alert_report["rules"] if row["missing_metric_names"] or row.get("seven_day_history_available") is False]
    display({"notification_mode": alert_report["notifications"], "coverage_gaps": gaps})
else:
    print("Preview mode: this verification/evidence step follows the explicitly executed workload.")


## 5a. Inspect real request traces

Jaeger is available at `http://127.0.0.1:16686` (or `JAEGER_PORT`). Grafana Explore also has a provisioned **Jaeger** datasource. The shared Milvus/WP configuration exports OTLP with full sampling for small local diagnostic runs. Read [TRACING.md](../TRACING.md) before sustained performance comparisons.

Enable the following optional 100-vector probe to save exact request trace IDs and raw spans. The supplied image currently shows a **partial** Insert trace: Proxy/coordinator spans are exported, but SN/WP do not remain under that user request ID. Query/search traces include SN and segcore. Separate WP traces do not prove a connected write chain.

Use `TRACES_REQUIRE_E2E=1 ./stack.sh traces-smoke` for a strict automated check that fails on these gaps. Jaeger history is bounded and in memory; save JSON evidence before eviction or restart.


In [ ]:
if EXECUTE:
    RUN_TRACE_SMOKE = False
    if RUN_TRACE_SMOKE:
        lab.run("traces-smoke")
        trace_report = json.loads((lab.state / "tracing-smoke.json").read_text())
        display({key: trace_report.get(key) for key in ("status", "services", "coverage_gaps", "failures")})
        for request in trace_report.get("requests", []):
            trace = request.get("trace", {})
            if trace.get("url"):
                display(HTML('<a target="_blank" href="' + html.escape(trace["url"]) + '">'
                             + html.escape(request["operation"]) + '</a>'))
    else:
        print("Optional trace probe skipped. Set RUN_TRACE_SMOKE=True to insert 100 vectors and inspect coverage.")

else:
    lab.run('traces-smoke')


## 5b. Verify collected container logs

Loki/Alloy is selected by this full walkthrough. The endpoint cell also opens or reuses its local forward.
In Grafana Explore select **Loki**, query `{namespace="milvus-stack"}`, then choose a Pod or filter text. Recognized `[traceID=...]` fields link to Jaeger when selected.

This smoke test writes 10 vectors, verifies query/search and matches the actual Proxy collection marker in Loki. Loki uses a 4 GiB PVC and 24-hour retention; Jaeger uses bounded memory and loses history on restart. Read [LOGGING.md](../LOGGING.md) and [TRACING.md](../TRACING.md).


In [ ]:
if EXECUTE:
    lab.run("logs-smoke")
    logging_report = json.loads((lab.state / "logging-smoke.json").read_text())
    display({key: logging_report.get(key) for key in ("passed", "query", "matched_lines", "failures")})

else:
    lab.run('logs-smoke')


## 6. Exercise the alert pipeline

The offline tests check 16 alert behaviors with the pinned Prometheus image.
The live probe creates temporary synthetic rules/events, verifies firing → AM receipt → recovery and inhibition, and cleans up afterward.
It takes several minutes because Kubernetes rule files reload asynchronously. You can watch Prometheus and AM while it runs.
It does not stop business Pods or send email. External notification configurations are refused by this local smoke probe.

In [ ]:
if EXECUTE:
    if RUN_RULE_TESTS:
        lab.run("alerts-test")
    else:
        print("Offline rule tests skipped.")
else:
    print("Preview mode: this verification/evidence step follows the explicitly executed workload.")


In [ ]:
if EXECUTE:
    if RUN_ALERT_PROBE:
        lab.run("alerts-smoke")
        probe = json.loads((lab.state / "alerting-smoke.json").read_text())
        display({key: probe.get(key) for key in ("pipeline_probe", "inhibition_probe", "failures")})
    else:
        print("Live alert probe skipped.")
else:
    lab.run('alerts-smoke')


## 7. Measure the real write path

Default measurement: 60 seconds, 2 workers, 100 rows/batch, 128 dimensions and a total target of 5 requests/s (500 rows/s).
Warm-up statistics are separate. This is a local REST/port-forward baseline, not a maximum-capacity result.
Change `PERF_SETTINGS` above to repeat the experiment; use hardware-specific throughput/latency gates when you have a baseline.

After writing, the runner checks strong-read counts, sampled IDs and filtered vector search. It collects metrics, AM snapshots and 10 business/dependency Pod logs even on handled failures.
These checks establish live visibility; they do not test crash/recovery durability. WP metrics aggregate the deployment rather than tracing each request end to end.

In [ ]:
perf_path = None
perf_report = None
if EXECUTE:
    perf_path = None
    perf_report = None
    if RUN_PERF:
        perf_rc = lab.run("perf", env=PERF_SETTINGS, check=False)
        perf_path = lab.latest_perf()
        perf_report = show_report(perf_path)
        assert perf_rc == 0, f"Performance run failed. Inspect {perf_path} and the evidence cells below."
    else:
        print("Performance run skipped. Existing reports remain under .local/perf/.")
else:
    lab.run('perf')


## 8. Inspect the evidence and slowest requests

Each `.local/perf/<run-id>/` directory includes `summary.md`, `report.json`, every insert attempt in `requests.jsonl`, metric ranges, snapshots, Pod status/events and bounded timestamped logs.
The report has dashboard links for the test time window. Local request IDs are not assumed to be server trace IDs; correlate collection metadata, timestamps, log identity and node labels.
Missing lazy error counters are coverage gaps. An empty `ALERTS` series can simply mean no pending/firing rules.

In [ ]:
if perf_path:
    display({"report": str(perf_path), "collection_info": perf_report.get("collection_info"),
             "correctness": perf_report.get("correctness"), "observations": perf_report.get("observations"),
             "diagnostic_errors": perf_report["diagnostic_errors"]})
    requests = [json.loads(line) for line in (perf_path.parent / "requests.jsonl").read_text().splitlines()]
    slowest = sorted(requests, key=lambda row: row["latency_ms"], reverse=True)[:5]
    display([{key: row.get(key) for key in ("request_id", "at", "phase", "latency_ms", "ok", "error")} for row in slowest])
    display({name: {key: row.get(key) for key in ("finite_samples", "max", "counter_delta")}
             for name, row in perf_report["metrics"].items()})
else:
    print("Run perf first to create a correlated experiment directory.")

In [ ]:
if perf_path:
    log_files = sorted(perf_path.parent.glob("*.log"))
    print("Log files:", [path.name for path in log_files])
    current_logs = [path for path in log_files if not path.name.endswith("-previous.log")]
    preferred = [path for path in current_logs if "woodpecker-0" in path.name]
    selected = next((path for path in preferred + current_logs if path.stat().st_size > 0), None)
    if selected:
        print("Last 20 lines:", selected.name)
        tail = "\n".join(selected.read_text(errors="replace").splitlines()[-20:])[-12000:]
        display(HTML("<pre>" + html.escape(tail) + "</pre>"))

## 9. Optional: demonstrate a failed performance gate

Set `RUN_GATE_DEMO=True` to write 1,000 additional rows at a small rate with an intentionally impossible 0.001-ms p99 gate.
The expected outcome is a nonzero status with successful writes/read checks and retained diagnostic evidence.
This demonstration does not inject a failure into Milvus or Woodpecker.

In [ ]:
if EXECUTE:
    if RUN_GATE_DEMO:
        gate_rc = lab.run("perf", env={**PERF_SETTINGS, "PERF_DURATION": "5", "PERF_WARMUP": "0",
                                      "PERF_RPS": "2", "PERF_MAX_P99_MS": "0.001"}, check=False)
        gate_path = lab.latest_perf()
        gate_report = show_report(gate_path)
        assert gate_rc != 0 and any("p99" in failure for failure in gate_report["failures"])
    else:
        print("Gate failure demonstration skipped. Set RUN_GATE_DEMO=True to exercise it.")
else:
    print("Preview mode: this verification/evidence step follows the explicitly executed workload.")


## 10. Collect wider diagnostics or repeat automated acceptance

`collect` includes all Pod logs, resource status, images and dashboard/alert reports under a timestamped `.local/evidence/` directory.
The optional `acceptance` command repeats the earlier rule/workload/dashboard/live-probe checks as one CLI entry point. It does not run perf or email tests by default.
Review application logs before sharing evidence.

In [ ]:
if EXECUTE:
    lab.run("collect")
    if RUN_ACCEPTANCE:
        lab.run("acceptance")
    else:
        print("Repeated acceptance skipped; CLI equivalent: ./stack.sh acceptance")
    print("Evidence directory:", lab.state / "evidence")
else:
    print("Preview mode: this verification/evidence step follows the explicitly executed workload.")


## 11. Pause/resume workloads or uninstall the stack

These cells remain disabled by default. `EXECUTE=True` is also required for resource changes.

| Action | Effect |
| --- | --- |
| `lab.close_forwards()` | Close only this notebook's forwards |
| `stop` / `start` | Pause/resume this stack's workloads on the existing Kubernetes cluster; retain releases/PVCs and restore saved replica counts |
| `uninstall` / `install` | Delete stack releases/PVC data, then install a fresh stack |

`stop` records the workload replica counts in a namespace ConfigMap before pausing them. `start` restores those counts. Memory/ephemeral history such as Jaeger traces is lost when its Pod exits. The cluster itself is managed externally.


In [ ]:
CLOSE_NOTEBOOK_FORWARDS = False
if CLOSE_NOTEBOOK_FORWARDS:
    lab.close_forwards()
    print("Notebook-owned forwards closed. External forwards were left alone.")
else:
    print("Keeping dashboard access available.")

In [ ]:
STOP_CLUSTER = False
if STOP_CLUSTER:
    lab.close_forwards()
    lab.run("stop")
else:
    print("Stack workloads remain running. To pause them, set STOP_CLUSTER=True; EXECUTE=True executes the plan.")

In [ ]:
RESUME_CLUSTER = False
if RESUME_CLUSTER:
    lab.run("start")
    lab.open_forwards()
else:
    print("Resume skipped. Set RESUME_CLUSTER=True when you want to resume this stack’s paused workloads.")

In [ ]:
UNINSTALL_STACK_AND_DATA = False
if UNINSTALL_STACK_AND_DATA:
    lab.close_forwards()
    lab.run("uninstall")
else:
    print("Uninstall skipped: all stack data is retained. Set UNINSTALL_STACK_AND_DATA=True only for a fresh reset.")

In [ ]:
REINSTALL_AFTER_UNINSTALL = False
if REINSTALL_AFTER_UNINSTALL:
    lab.run("install")
    lab.open_forwards()
else:
    print("Fresh redeployment skipped.")

## References

- [Stack README](../README.md): deployment, commands and dashboard sources.
- [Performance guide](../PERFORMANCE.md): load settings, gates, evidence schema and investigation workflow.
- [Alerting guide](../ALERTING.md): rule catalogue, coverage gaps and optional SMTP examples.

Notebook outputs are local observations. Clear all outputs before sharing/committing an edited source notebook;
the shipped notebook contains no executed output, credentials or machine-specific paths.